# DR Model Training with Acquisition Artifacts

Baseline experiment for **Exposing Dataset Artifacts in Medical AI**. The model is trained on the original APTOS images without artifact-removal preprocessing.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, random
import pandas as pd
from tqdm import tqdm
BASE_DIR='/content/drive/MyDrive/Fundus_Artifact_Project'
CSV_PATH=os.path.join(BASE_DIR,'APTOS_2019','train.csv')
IMG_DIR=os.path.join(BASE_DIR,'APTOS_2019','train_images')
RAW_DATA=os.path.join(BASE_DIR,'Raw_DR_Classifier')
RESULTS_DIR=os.path.join(BASE_DIR,'Results','raw_image_model')
os.makedirs(RESULTS_DIR,exist_ok=True)


In [ ]:
# Construct binary DR dataset: diagnosis 0 = no_dr; diagnosis 1-4 = dr
df=pd.read_csv(CSV_PATH)
df['label_binary']=df['diagnosis'].apply(lambda x:'dr' if int(x)>0 else 'no_dr')
for label in ['dr','no_dr']:
    folder=os.path.join(RAW_DATA,label)
    os.makedirs(folder,exist_ok=True)
for _,row in tqdm(df.iterrows(),total=len(df)):
    src=os.path.join(IMG_DIR,row['id_code']+'.png')
    dst=os.path.join(RAW_DATA,row['label_binary'],row['id_code']+'.png')
    if os.path.isfile(src) and not os.path.isfile(dst): shutil.copy2(src,dst)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms,models
from torch.utils.data import DataLoader,random_split
IMG_SIZE=224; BATCH_SIZE=32; SEED=42
transform=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
dataset=datasets.ImageFolder(RAW_DATA,transform=transform)
generator=torch.Generator().manual_seed(SEED)
n_train=int(0.8*len(dataset)); n_val=len(dataset)-n_train
train_set,val_set=random_split(dataset,[n_train,n_val],generator=generator)
train_loader=DataLoader(train_set,batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_set,batch_size=BATCH_SIZE,shuffle=False)
print('Class mapping:',dataset.class_to_idx)


In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc=nn.Linear(model.fc.in_features,2); model=model.to(device)
criterion=nn.CrossEntropyLoss(); optimizer=optim.Adam(model.parameters(),lr=1e-4)
EPOCHS=10; history=[]
for epoch in range(EPOCHS):
    model.train(); train_correct=train_total=0
    for x,y in train_loader:
        x,y=x.to(device),y.to(device); optimizer.zero_grad(); out=model(x); loss=criterion(out,y); loss.backward(); optimizer.step()
        train_correct+=(out.argmax(1)==y).sum().item(); train_total+=y.size(0)
    model.eval(); val_correct=val_total=0
    with torch.no_grad():
        for x,y in val_loader:
            x,y=x.to(device),y.to(device); out=model(x); val_correct+=(out.argmax(1)==y).sum().item(); val_total+=y.size(0)
    row={'epoch':epoch+1,'train_accuracy':train_correct/train_total,'val_accuracy':val_correct/val_total}; history.append(row); print(row)
    torch.save(model.state_dict(),os.path.join(RESULTS_DIR,f'epoch_{epoch+1}.pt'))
torch.save(model.state_dict(),os.path.join(RESULTS_DIR,'dr_classifier_resnet18.pt'))
pd.DataFrame(history).to_csv(os.path.join(RESULTS_DIR,'training_history.csv'),index=False)


## Comparison role

This model is the **artifact-retaining baseline**. It should be compared against the model trained after the controlled preprocessing stage, using the same task definition and external-validation protocol.
